## Topic: Pinecone Vector Database

### Agenda
- 1. Phase 1 — Pinecone Fundamentals

- 2. Phase 2 — Pinecone Setup & First Connection

- 3. Phase 3 — Pinecone Core Architecture

- 4. Phase 4 — First UPSERT (Insert Vectors into Pinecone) and Vector Operations

### 1. Phase 1 — Pinecone Fundamentals

- Definition:
    - Pinecone is a fully managed, cloud-based Vector Database.

    - Pinecone is a managed vector database platform designed to store, index, and retrieve high-dimensional vectors efficiently.


### 2. Phase 2 — Pinecone Setup & First Connection

In [ ]:
""" 
1. Create Your Free Pinecone Account
    STEP 1: Open https://www.pinecone.io/
    STEP 2: Click "Sign Up Free" (Starter plan — no credit card)
    STEP 3: Verify email (check inbox/spam)
    STEP 4: Log into Console

"""

In [ ]:
""" 
2. Generate Your API Key
In Pinecone Console:
  [Left Menu] → API Keys → [Copy] → Store in secure note/text file

FORMAT LOOKS LIKE:
  "pcsk_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"

"""

In [ ]:
""" 
3. Install SDK Packages

- 1. Install Pinecone Python SDK
    - pip install pinecone

    - pip show pinecone

- 2. Install LangChain Pinecone Integration
    - pip install langchain-pinecone

    - pip show langchain-pinecone


- 3. Full Package List (Install All at Once) 
    - pip install pinecone langchain-pinecone langchain-ollama

    - where,
        - pinecone	: Official Pinecone SDK (v5+) — index management.
        - langchain-pinecone : PineconeVectorStore for LangChain.
        - langchain-ollama	: Connect local LLM + embeddings

    - here,
        -  Check Ollama is Running (If Using Local LLM)
        - 1. http://localhost:11434 (see a simple text message: "Ollama is running)
        - 2. ollama ls (see the model details)

"""

In [1]:
# Run this quick Python check:
import pinecone
from langchain_pinecone import PineconeVectorStore

print("Pinecone SDK version:", pinecone.__version__)
print("LangChain Pinecone:", PineconeVectorStore.__name__)

# for upgrade
# pip install --upgrade pinecone langchain-pinecone

c:\Users\kz\anaconda3\envs\llmapp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Pinecone SDK version: 7.3.0
LangChain Pinecone: PineconeVectorStore


#### Connects __ Pinecone API key

In [2]:
# verify_connection.py
import os
from pinecone import Pinecone

# 1. Load API key from environment
api_key = os.getenv("PINECONE_API_KEY")
if not api_key:
    raise ValueError("PINECONE_API_KEY not set! Run: export PINECONE_API_KEY='...'")

# 2. Initialize modern SDK client
pc = Pinecone(api_key=api_key)

# 3. List existing indexes (proves connection works)
indexes = pc.list_indexes()
print("Connected! Your Pinecone indexes:")
print(indexes)


Connected! Your Pinecone indexes:
[]


### 3. Phase 3 — Pinecone Core Architecture

In [ ]:
"""
        - Pinecone's Core Architecture
        ===============================

Pinecone
   │
   └── Index
        │
        ├── Namespace
        │     │
        │     ├── Vector
        │     │    ├── ID
        │     │    ├── Values
        │     │    └── Metadata
        │     │
        │     └── Vector
        │
        └── Namespace

        
- What is Pinecone?	
        - A fully managed cloud vector database. 
        - It stores embeddings (vectors) + metadata and performs similarity search over the internet via API.

        
- What Is a Pinecone Index?
        - An index is the primary structure where your vectors are stored and searched.

        - The container inside Pinecone where vectors + metadata are stored. Think of it like a "Table" in SQL or a .index file in FAISS. You create ONE index per project/dataset.

- Critical rule
        - The embedding model dimension and Pinecone index dimension must match.
        - Otherwise, the vector cannot be inserted correctly.

- What is Namespace?
        - A namespace in a Pinecone database is a logical partition inside a single index that lets you split and isolate your vector data.

        - example:
        company-rag-index
                │
                ├── hr
                │    ├── vector-001
                │    └── vector-002
                │
                ├── engineering
                │    ├── vector-003
                │    └── vector-004
                │
                └── finance
                ├── vector-005
                └── vector-006

- What Is a Vector?
        - A vector is simply an array of numerical values representing some content.


- ID: 
        - The id uniquely identifies a vector record.
        - For document applications, IDs are often generated systematically.

- Values:
        - values are the actual embedding numbers.
        - values = [0.123,-0.421,0.732,0.091,...]

- Metadata: 
        - Metadata contains additional information about the vector.

"""

In [ ]:
"""   - Anatomy of a Vector in Pinecone
    ========================================
{
    "id": "doc-001",
    "values": [0.12, -0.42, 0.78, ...],
    "metadata": {
        "source": "ml_book.pdf",
        "page": 15,
        "topic": "machine-learning"
    }
}

"""

- At a glance: 

| Term          | Meaning                                     |
| ------------- | ------------------------------------------- |
| **Pinecone**  | Managed vector database platform            |
| **Index**     | Main searchable vector structure            |
| **Namespace** | Logical partition within an index           |
| **Vector**    | Numerical representation of content         |
| **ID**        | Unique identifier for a vector              |
| **Values**    | Actual embedding numbers                    |
| **Metadata**  | Additional information attached to a vector |
| **Query**     | Search for similar vectors                  |
| **Upsert**    | Insert or update vectors                    |
| **Fetch**     | Retrieve vectors by ID                      |
| **Delete**    | Remove vectors                              |




In [ ]:
"""  - Complete Pinecone Data Flow
==================================

                    DOCUMENT
                       │
                       ↓
                Embedding Model
                       │
                       ↓
                     Vector
                       │
          ┌────────────┴────────────┐
          ↓                         ↓
         ID                      Metadata
          │                         │
          └────────────┬────────────┘
                       ↓
                  Pinecone
                       │
                    Index
                       │
                  Namespace
                       │
                       ↓
                  Stored Vector


"""

### 4. Phase 4 — First UPSERT (Insert Vectors into Pinecone)


In [ ]:
""" 
            Text
              ↓
            Embedding Model
              ↓
            Embedding Vector
              ↓
            Pinecone
              ↓
            UPSERT
              ↓
            Stored Vector

"""

#### Step1: Create index 

In [ ]:

import os
from dotenv import load_dotenv
from pinecone import Pinecone

load_dotenv()
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY")) 

# pc = Pinecone(
#     api_key=os.environ["PINECONE_API_KEY"]
# )

# Create a dense index with integrated embedding
index_name = "pinecone-learning"
if not pc.has_index(index_name):
    pc.create_index_for_model(
        name = index_name,
        cloud="aws",
        region="us-east-1", # aws account region

        embed={
            "model":"llama-text-embed-v2", # embedding model -> vector dimension: 1024
            "field_map":{"text": "chunk_text"} # chunking 
        }
    )

#### Step 2: insert data 

In [6]:
# step 2.1: Crete a some Dataset
 
records = [
    { "_id": "rec1", "chunk_text": "The Eiffel Tower was completed in 1889 and stands in Paris, France.", "category": "history" },
    { "_id": "rec2", "chunk_text": "Photosynthesis allows plants to convert sunlight into energy.", "category": "science" },
    { "_id": "rec3", "chunk_text": "Albert Einstein developed the theory of relativity.", "category": "science" },
    { "_id": "rec4", "chunk_text": "The mitochondrion is often called the powerhouse of the cell.", "category": "biology" },
    { "_id": "rec5", "chunk_text": "Shakespeare wrote many famous plays, including Hamlet and Macbeth.", "category": "literature" },
    { "_id": "rec6", "chunk_text": "Water boils at 100°C under standard atmospheric pressure.", "category": "physics" },
    { "_id": "rec7", "chunk_text": "The Great Wall of China was built to protect against invasions.", "category": "history" },
    { "_id": "rec8", "chunk_text": "Honey never spoils due to its low moisture content and acidity.", "category": "food science" },
    { "_id": "rec9", "chunk_text": "The speed of light in a vacuum is approximately 299,792 km/s.", "category": "physics" },
    { "_id": "rec10", "chunk_text": "Newton’s laws describe the motion of objects.", "category": "physics" },
    { "_id": "rec11", "chunk_text": "The human brain has approximately 86 billion neurons.", "category": "biology" },
    { "_id": "rec12", "chunk_text": "The Amazon Rainforest is one of the most biodiverse places on Earth.", "category": "geography" },
    { "_id": "rec13", "chunk_text": "Black holes have gravitational fields so strong that not even light can escape.", "category": "astronomy" },
    { "_id": "rec14", "chunk_text": "The periodic table organizes elements based on their atomic number.", "category": "chemistry" },
    { "_id": "rec15", "chunk_text": "Leonardo da Vinci painted the Mona Lisa.", "category": "art" },
    { "_id": "rec16", "chunk_text": "The internet revolutionized communication and information sharing.", "category": "technology" },
    { "_id": "rec17", "chunk_text": "The Pyramids of Giza are among the Seven Wonders of the Ancient World.", "category": "history" },
    { "_id": "rec18", "chunk_text": "Dogs have an incredible sense of smell, much stronger than humans.", "category": "biology" },
    { "_id": "rec19", "chunk_text": "The Pacific Ocean is the largest and deepest ocean on Earth.", "category": "geography" },
    { "_id": "rec20", "chunk_text": "Chess is a strategic game that originated in India.", "category": "games" },
    { "_id": "rec21", "chunk_text": "The Statue of Liberty was a gift from France to the United States.", "category": "history" },
    { "_id": "rec22", "chunk_text": "Coffee contains caffeine, a natural stimulant.", "category": "food science" },
    { "_id": "rec23", "chunk_text": "Thomas Edison invented the practical electric light bulb.", "category": "inventions" },
    { "_id": "rec24", "chunk_text": "The moon influences ocean tides due to gravitational pull.", "category": "astronomy" },
    { "_id": "rec25", "chunk_text": "DNA carries genetic information for all living organisms.", "category": "biology" },
    { "_id": "rec26", "chunk_text": "Rome was once the center of a vast empire.", "category": "history" },
    { "_id": "rec27", "chunk_text": "The Wright brothers pioneered human flight in 1903.", "category": "inventions" },
    { "_id": "rec28", "chunk_text": "Bananas are a good source of potassium.", "category": "nutrition" },
    { "_id": "rec29", "chunk_text": "The stock market fluctuates based on supply and demand.", "category": "economics" },
    { "_id": "rec30", "chunk_text": "A compass needle points toward the magnetic north pole.", "category": "navigation" },
    { "_id": "rec31", "chunk_text": "The universe is expanding, according to the Big Bang theory.", "category": "astronomy" },
    { "_id": "rec32", "chunk_text": "Elephants have excellent memory and strong social bonds.", "category": "biology" },
    { "_id": "rec33", "chunk_text": "The violin is a string instrument commonly used in orchestras.", "category": "music" },
    { "_id": "rec34", "chunk_text": "The heart pumps blood throughout the human body.", "category": "biology" },
    { "_id": "rec35", "chunk_text": "Ice cream melts when exposed to heat.", "category": "food science" },
    { "_id": "rec36", "chunk_text": "Solar panels convert sunlight into electricity.", "category": "technology" },
    { "_id": "rec37", "chunk_text": "The French Revolution began in 1789.", "category": "history" },
    { "_id": "rec38", "chunk_text": "The Taj Mahal is a mausoleum built by Emperor Shah Jahan.", "category": "history" },
    { "_id": "rec39", "chunk_text": "Rainbows are caused by light refracting through water droplets.", "category": "physics" },
    { "_id": "rec40", "chunk_text": "Mount Everest is the tallest mountain in the world.", "category": "geography" },
    { "_id": "rec41", "chunk_text": "Octopuses are highly intelligent marine creatures.", "category": "biology" },
    { "_id": "rec42", "chunk_text": "The speed of sound is around 343 meters per second in air.", "category": "physics" },
    { "_id": "rec43", "chunk_text": "Gravity keeps planets in orbit around the sun.", "category": "astronomy" },
    { "_id": "rec44", "chunk_text": "The Mediterranean diet is considered one of the healthiest in the world.", "category": "nutrition" },
    { "_id": "rec45", "chunk_text": "A haiku is a traditional Japanese poem with a 5-7-5 syllable structure.", "category": "literature" },
    { "_id": "rec46", "chunk_text": "The human body is made up of about 60% water.", "category": "biology" },
    { "_id": "rec47", "chunk_text": "The Industrial Revolution transformed manufacturing and transportation.", "category": "history" },
    { "_id": "rec48", "chunk_text": "Vincent van Gogh painted Starry Night.", "category": "art" },
    { "_id": "rec49", "chunk_text": "Airplanes fly due to the principles of lift and aerodynamics.", "category": "physics" },
    { "_id": "rec50", "chunk_text": "Renewable energy sources include wind, solar, and hydroelectric power.", "category": "energy" }
]


In [7]:
# step 2.2: Connect to Pinecone Index
dense_index = pc.Index(index_name)

c:\Users\kz\anaconda3\envs\llmapp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# step 2.3: Perform the  Upsert the records into a namespace (convert record to embedding)
dense_index.upsert_records("example-namespace", records)

In [9]:
# step 2.4: Check the status
# Wait for the upserted vectors to be indexed
import time
time.sleep(10)

# View stats for the index
stats = dense_index.describe_index_stats()
print(stats)



{'dimension': 1024,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'example-namespace': {'vector_count': 50}},
 'total_vector_count': 50,
 'vector_type': 'dense'}


#### Step 3: Fetch the Data From Pinecone

In [10]:
# Define the query
query = "Famous historical structures and monuments"

# Search the dense index
results = dense_index.search(
    namespace="example-namespace",
    query={
        "top_k": 5,
        "inputs": {
            'text': query
        }
    }
)

# Print the results
for hit in results['result']['hits']:
    print(f"id: {hit['_id']}, score: {round(hit['_score'], 2)}, text: {hit['fields']['chunk_text']}, category: {hit['fields']['category']}")


id: rec17, score: 0.28, text: The Pyramids of Giza are among the Seven Wonders of the Ancient World., category: history
id: rec38, score: 0.19, text: The Taj Mahal is a mausoleum built by Emperor Shah Jahan., category: history
id: rec5, score: 0.18, text: Shakespeare wrote many famous plays, including Hamlet and Macbeth., category: literature
id: rec15, score: 0.1, text: Leonardo da Vinci painted the Mona Lisa., category: art
id: rec50, score: 0.09, text: Renewable energy sources include wind, solar, and hydroelectric power., category: energy


#### Step 4: Delete the index

In [11]:
# Delete the index
pc.delete_index(index_name)